# Saldo Disponible Real 2024 – 2026 · TATSA y TVL

Calcula el saldo disponible del presupuesto por **SKU + centro de costo + año**, con el mismo formato del Excel `Saldo_Disponible_Odoo_PQ`, y un informe de **RQ pendientes** (con OC y sin OC).

**Saldo = Presupuesto − (Compras + RQ_COC + RQ_SOC + Gasto_Sin_OC)**

| Columna | Qué es | De dónde sale |
|---|---|---|
| Presupuesto | Presupuesto vigente (ya incluye redistribuciones) | `crossovered.budget.lines.planned_amount` agrupado con `read_group` (trae todos los centros) |
| Compras | Lo ejecutado con OC: bienes = recibido neto de devoluciones; servicios = facturado | Líneas de OC confirmadas |
| RQ | Valor total pedido en OC confirmadas | Líneas de OC confirmadas |
| RQ_COC | En tránsito **con OC**: pedido − ejecutado | Líneas de OC abiertas |
| RQ_SOC | En tránsito **sin OC**: RQ aprobado − lo ya pedido en OC | Líneas de RQ abiertas |
| RQ_COC_und / RQ_SOC_und | Unidades en tránsito con OC / sin OC (UdM del producto). En servicios, cantidad pendiente de facturar (referencial) | Mismas líneas |
| Gasto_Sin_OC | Facturas de proveedor sin OC (rendiciones, fondo fijo, gasto directo) | `account.move.line` sin `purchase_line_id` |
| Gasto_Rendicion | Parte del gasto sin OC que está dentro de una rendición | `account.surrender.requests.it` |
| Facturado_Odoo | Lo que Odoo resta en la redistribución ("Facturado Anual"): todas las facturas de proveedor del SKU + centro, con IGV, por **año de emisión** | `account.move.line` |
| Disponible_Odoo | Lo que Odoo deja redistribuir: Presupuesto − Facturado_Odoo | |
| Disponible_Redistribuir | Lo que realmente se puede ceder: el Saldo si es positivo, si no 0 | |
| Diferencia_vs_Odoo | Disponible_Odoo − Saldo. Positivo = Odoo deja redistribuir más de lo real | |

La diferencia se explica en: **RQ_SOC** y **RQ_COC** (comprometido que Odoo no ve), **Dif_Factura_Otro_Anio** (factura emitida en otro año que el del RQ, típico de importaciones), **Dif_Recibido_vs_Facturado** (recibido sin factura, o facturado antes de llegar) y **Dif_Otros** (gasto sin OC por fecha contable vs emisión, centro distinto en factura y OC, redondeos) y **Dif_NC_Devolucion** (nota de crédito por devolución de una compra con OC, que Odoo resta en el año de la NC).

Reglas:
- **Situación presupuestaria:** Odoo mide el consumo por **SKU + centro** (las columnas de consumo son iguales en todas las situaciones del mismo SKU y centro). Por eso, si un SKU tiene varias situaciones en el mismo centro, **se suma su presupuesto** y la compra se cuenta **una sola vez**. Se informa `Partida` (grupo de la situación, para segmentar), `Situación presupuestaria` (unidas con " + " si hay varias) y `Ppto_por_Situacion` (cuánto aporta cada una, para saber de dónde redistribuir).
- **Visible_Odoo:** "Sí" si la línea de presupuesto aparece con la búsqueda normal de Odoo del `USUARIO` (lo que se veía inicialmente; el módulo `presupuesto_view_details` oculta los centros administrativos). Para filas sin presupuesto se usa el centro.
- **Año:** el del periodo de presupuesto del RQ (`periodo_id`); si la OC no tiene RQ, el año de aprobación de la OC. El gasto sin OC va por fecha contable. Así una importación facturada por el proveedor en otro año no se cuela en el año equivocado (caso G1010460).
- **Montos con IGV** según los impuestos reales de cada línea (como calcula Odoo).
- **Bienes:** lo facturado se valora con la factura (precio y TC de la factura, igual que el kardex ya regularizado); lo recibido sin factura y lo pendiente, con el precio y la tasa de la OC. No depende del kardex, así que un albarán con TC = 1 o un GV pendiente no lo altera.
- **Gastos vinculados (GV)** de importación no se suman al repuesto: ya se restan en sus propios SKU (G326xxxx) como gasto.
- **Unidades:** todo se lleva a la UdM del producto. Si el RQ, la OC o la recepción usan otra UdM se convierte y se anota en *Observaciones*.
- **Devoluciones a proveedor** se restan de lo recibido de la OC, en el **año del RQ** (la compra original). La nota de crédito del proveedor que no está ligada a la OC **no** se vuelve a restar como gasto (si no, el mismo monto se liberaría dos veces y en el año de la NC). Odoo sí la cuenta en el año de la NC: aparece en `Dif_NC_Devolucion`.
- Las columnas calculadas pesadas del presupuesto (`s_previsto`, `s_ordenado`, `planned_amount_computed`…) **no se usan**.
- Solo lectura: el notebook no escribe nada en Odoo.

##### Librerías

In [ ]:
import xmlrpc.client # Comunicar con Odoo
import pandas as pd
import numpy as np
import os
from collections import defaultdict
from dotenv import load_dotenv
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

##### Parámetros

In [ ]:
ANIOS = [2024, 2025, 2026]                  # Años a calcular
EMPRESAS = {"TATSA": 2, "TVL": 1}           # Nombre -> id de compañía en Odoo
USUARIO = "JIM"                             # Usuario del .env (JIM, LUIS o ANTONIO). LUIS no puede leer presupuesto
FECHA_DESDE = "2023-07-01"                  # Ventana para traer RQ/OC/facturas que pueden caer en los años pedidos
TOLERANCIA = 0.0001                         # Cantidades menores se consideran cero
PRODUCTOS_EXCLUIDOS = {"ANTICIPO"}          # Líneas de factura que no son consumo de presupuesto
SOLO_VISIBLES_RQ_SIN_OC = True             # En el Excel, RQ sin OC solo de centros visibles en Odoo
UMBRAL_ALERTA = 100.0                       # Diferencias menores (S/) no se listan en alertas de redistribución

EXPORTAR_EXCEL = True                    # Cambiar a True para generar el Excel
ARCHIVO_EXCEL = "Saldo_Disponible_Real_2024_2026.xlsx"

##### Credenciales y conexión

In [ ]:
"""
    1. CONECTAR A ODOO - SE CREA OBJECTO PARA IDENTIFICAR + OBJECTO PARA HACER CONSULTAS
"""
load_dotenv() # Cargar las variables de entorno desde el archivo .env
url = os.getenv("URL_ODOO")
db = os.getenv("BD_ODOO")
username = os.getenv(f"USER_{USUARIO}")
password = os.getenv(f"PASSWORD_{USUARIO}")

common = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common")
uid = common.authenticate(db, username, password, {})
print(f"uid conectado : {uid}")

models = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object", allow_none=True)


"""
    2. FUNCIONES DE LECTURA (SOLO LECTURA)
"""
def odoo(modelo, metodo, argumentos, opciones=None, empresas=None):
    # Equivalente a models.execute_kw con el contexto de idioma, zona horaria y compañías
    opciones = dict(opciones or {})
    contexto = {"lang": "es_PE", "tz": "America/Lima", **opciones.pop("context", {})}
    if empresas:
        contexto["allowed_company_ids"] = empresas
    opciones["context"] = contexto
    return models.execute_kw(db, uid, password, modelo, metodo, argumentos, opciones)


def leer_todo(modelo, dominio, campos, empresas, lote=5000, contexto=None):
    # search_read por bloques para no saturar la respuesta
    filas, desplazamiento = [], 0
    while True:
        bloque = odoo(modelo, "search_read", [dominio],
                      {"fields": campos, "limit": lote, "offset": desplazamiento, "order": "id", "context": contexto or {}},
                      empresas)
        filas += bloque
        if len(bloque) < lote:
            return filas
        desplazamiento += lote


def leer_ids(modelo, ids, campos, empresas, lote=2000):
    # read por bloques de ids
    ids, filas = list(ids), []
    for i in range(0, len(ids), lote):
        filas += odoo(modelo, "read", [ids[i:i + lote]], {"fields": campos}, empresas)
    return filas


def m2o_id(valor):
    return valor[0] if isinstance(valor, list) and valor else None


def m2o_nombre(valor):
    return valor[1] if isinstance(valor, list) and len(valor) == 2 else None

##### Tablas maestras: llaves, productos, unidades y años fiscales

In [ ]:
"""
    1. LLAVES DE GRUPO DE SKU
"""
df_llaves = pd.read_excel("Bd_Llaves_Actualizado.xlsx", engine="calamine")
df_llaves = df_llaves[["Cod_Producto", "Cod_Grupo_Sku"]].dropna(subset=["Cod_Producto"]).drop_duplicates("Cod_Producto")


"""
    2. PRODUCTOS (product.product, incluye archivados)
"""
productos = leer_todo(
    "product.product", [],
    ["default_code", "name", "numero_parte", "product_brand_id", "categ_id", "detailed_type", "uom_id"],
    [1, 2], contexto={"active_test": False}
)
df_productos = pd.DataFrame([{
    "product_id": p["id"],
    "Cod_Producto": p["default_code"] or None,
    "Producto": p["name"],
    "Nro_Parte": p.get("numero_parte") or None,
    "Marca": m2o_nombre(p.get("product_brand_id")),
    "Familia": (m2o_nombre(p["categ_id"]) or "").split("/")[0].strip() or None,
    "Tipo_Producto": p["detailed_type"],
    "udm_producto": m2o_id(p["uom_id"]),
} for p in productos])
df_productos = df_productos.merge(df_llaves, on="Cod_Producto", how="left")
df_productos["Cod_Grupo_Sku"] = df_productos["Cod_Grupo_Sku"].fillna(df_productos["Cod_Producto"])
PRODUCTO = df_productos.set_index("product_id").to_dict("index")


"""
    3. UNIDADES DE MEDIDA (para convertir todo a la UdM del producto)
"""
UDM = {u["id"]: {"nombre": u["name"], "factor": u["factor"] or 1.0, "categoria": m2o_id(u["category_id"])}
       for u in odoo("uom.uom", "search_read", [[]], {"fields": ["name", "factor", "category_id"], "context": {"active_test": False}})}


"""
    4. AÑOS FISCALES (id -> año)
"""
ANIO_FISCAL = {a["id"]: int(a["name"]) for a in odoo("account.fiscal.year", "search_read", [[]], {"fields": ["name"]})
               if str(a["name"]).isdigit()}


"""
    5. ETIQUETAS DE ESTADOS (valor técnico -> lo que se ve en Odoo)
"""
ESTADO_OC = dict(odoo("purchase.order", "fields_get", [["state_new"]], {"attributes": ["selection"]})["state_new"]["selection"])
ESTADO_RQ = dict(odoo("purchase.requisition", "fields_get", [["state"]], {"attributes": ["selection"]})["state"]["selection"])

print(f"Productos: {len(df_productos):,} | UdM: {len(UDM)} | Años fiscales: {sorted(ANIO_FISCAL.values())}")

##### Funciones de conversión

In [ ]:
def a_udm_producto(cantidad, udm_origen, product_id):
    """
        Convierte una cantidad a la UdM del producto (misma regla que Odoo: cantidad / factor_origen * factor_destino).
        Devuelve (cantidad_convertida, observación).
    """
    udm_destino = PRODUCTO.get(product_id, {}).get("udm_producto")
    if not udm_origen or not udm_destino or udm_origen == udm_destino:
        return cantidad, ""
    origen, destino = UDM.get(udm_origen), UDM.get(udm_destino)
    if origen is None or destino is None:
        return cantidad, "UdM desconocida"
    if origen["categoria"] != destino["categoria"]:
        return cantidad, f"UdM de otra categoría ({origen['nombre']} vs {destino['nombre']}), no se pudo convertir"
    return cantidad / origen["factor"] * destino["factor"], f"Convertido de {origen['nombre']} a {destino['nombre']}"


def unir_observaciones(serie):
    # Junta observaciones únicas en un texto
    vistas = []
    for texto in serie.dropna():
        for parte in str(texto).split(" | "):
            if parte and parte not in vistas:
                vistas.append(parte)
    return " | ".join(vistas)

##### 1. Presupuesto vigente

Se agrupa con `read_group` porque la búsqueda normal de líneas de presupuesto está modificada (módulo `presupuesto_view_details`) y esconde los centros administrativos. `planned_amount` ya trae las redistribuciones.

In [ ]:
def clave_id(valor):
    # Id de Odoo como entero (None si viene vacío o NaN) para cruzar sin problemas de tipo
    return None if valor is None or (isinstance(valor, float) and np.isnan(valor)) or valor is False else int(valor)


def leer_presupuesto(empresa_id, anio):
    """
        1. AGRUPAR LÍNEAS DE PRESUPUESTO DEL AÑO POR SKU + CENTRO + SITUACIÓN (read_group trae todos los centros)
    """
    dominio = [
        ["company_id", "=", empresa_id],
        ["crossovered_budget_id.periodo_id.fiscal_year_id.name", "=", str(anio)],
        ["crossovered_budget_id.state", "not in", ["cancel", "draft"]],
    ]
    grupos = odoo("crossovered.budget.lines", "read_group",
                  [dominio, ["planned_amount"], ["product_id", "analytic_account_id", "general_budget_id"]],
                  {"lazy": False}, [empresa_id])
    df = pd.DataFrame([{
        "product_id": m2o_id(g["product_id"]),
        "centro_id": m2o_id(g["analytic_account_id"]),
        "situacion_id": m2o_id(g["general_budget_id"]),
        "Cuenta Analítica": m2o_nombre(g["analytic_account_id"]),
        "Situación presupuestaria": m2o_nombre(g["general_budget_id"]),
        "Presupuesto": g["planned_amount"] or 0.0,
    } for g in grupos])
    if df.empty:
        return df

    """
        2. LÍNEAS QUE SE VEN CON LA BÚSQUEDA NORMAL DE ODOO (LO QUE SE VEÍA INICIALMENTE)
    """
    ids = odoo("crossovered.budget.lines", "search", [dominio], {}, [empresa_id])
    visibles = {(clave_id(m2o_id(l["product_id"])), clave_id(m2o_id(l["analytic_account_id"])), clave_id(m2o_id(l["general_budget_id"])))
                for l in leer_ids("crossovered.budget.lines", ids, ["product_id", "analytic_account_id", "general_budget_id"], [empresa_id])}
    df["Visible_Odoo"] = [(clave_id(p), clave_id(c), clave_id(s)) in visibles
                          for p, c, s in zip(df["product_id"], df["centro_id"], df["situacion_id"])]
    df["Periodo"] = anio
    return df


presupuesto, CENTROS_VISIBLES = {}, {}
for nombre, empresa_id in EMPRESAS.items():
    presupuesto[nombre] = pd.concat([leer_presupuesto(empresa_id, anio) for anio in ANIOS], ignore_index=True)
    if presupuesto[nombre].empty:
        raise ValueError(f"{nombre}: no se leyó presupuesto. Revisar USUARIO (LUIS no puede leer presupuesto).")
    CENTROS_VISIBLES[nombre] = {clave_id(c) for c in presupuesto[nombre].loc[presupuesto[nombre]["Visible_Odoo"], "centro_id"].dropna()}
    print(nombre, presupuesto[nombre].groupby("Periodo")["Presupuesto"].sum().round(0).to_dict(),
          f"| líneas visibles en Odoo: {presupuesto[nombre]['Visible_Odoo'].mean():.0%} | centros visibles: {len(CENTROS_VISIBLES[nombre])}")


##### 1b. Situación presupuestaria por SKU + centro

Odoo controla el presupuesto por **SKU + centro**: si el SKU tiene varias situaciones en el mismo centro, el RQ y las compras se descuentan una sola vez del total. Aquí se arma, por año + SKU + centro:
- `Partida`: grupo de la situación (primer nivel, ej. REPUESTOS, OTROS GASTOS, GASTOS DE TI), para segmentar.
- `Situación presupuestaria`: situación completa; si hay varias, unidas con " + ".
- `Ppto_por_Situacion`: presupuesto de cada situación cuando hay más de una (de dónde sale el monto al redistribuir).
- `Varias_Situaciones` y `Visible_Odoo` (alguna de sus líneas se ve con la búsqueda normal de Odoo).


In [ ]:
CLAVES = ["Periodo", "product_id", "centro_id"]


def claves_float(df):
    # Claves numéricas homogéneas para cruzar sin problemas de tipo (los vacíos quedan NaN)
    df = df.copy()
    for clave in CLAVES:
        df[clave] = pd.to_numeric(df[clave], errors="coerce").astype("float64")
    return df


def partida(situacion):
    # Primer nivel de la situación presupuestaria ("OTROS GASTOS / MATERIALES PROYECTO" -> "OTROS GASTOS")
    return situacion.split(" / ")[0].strip() if isinstance(situacion, str) and situacion else None


def armar_info_situacion(nombre):
    """
        UNA FILA POR AÑO + SKU + CENTRO CON EL PRESUPUESTO SUMADO Y SUS SITUACIONES
    """
    filas = []
    for (anio, producto, centro), g in presupuesto[nombre].groupby(CLAVES, dropna=False):
        por_situacion = g.groupby("Situación presupuestaria", dropna=False)["Presupuesto"].sum()
        situaciones = [s for s in por_situacion.index if isinstance(s, str)]
        filas.append({
            "Periodo": anio, "product_id": producto, "centro_id": centro,
            "Presupuesto": g["Presupuesto"].sum(),
            "Partida": " + ".join(sorted({partida(s) for s in situaciones})),
            "Situación presupuestaria": " + ".join(situaciones),
            "Ppto_por_Situacion": " | ".join(f"{s}: {v:,.2f}" for s, v in por_situacion.items()) if len(situaciones) > 1 else "",
            "Varias_Situaciones": "Sí" if len(situaciones) > 1 else "No",
            "Visible_Odoo": bool(g["Visible_Odoo"].any()),
        })
    return claves_float(pd.DataFrame(filas))


def juntar_obs(a, b):
    return " | ".join(x for x in [a, b] if isinstance(x, str) and x)


INFO_SITUACION = {nombre: armar_info_situacion(nombre) for nombre in EMPRESAS}
for nombre, df in INFO_SITUACION.items():
    print(f"{nombre}: {len(df):,} SKU+centro+año | con varias situaciones: {(df['Varias_Situaciones'] == 'Sí').sum()} | partidas: {df['Partida'].nunique()}")


##### 2. Requerimientos y sus líneas

In [ ]:
def leer_requerimientos(empresa_id):
    """
        1. CABECERAS DE RQ (AÑO = PERIODO DE PRESUPUESTO DEL RQ; SI NO TIENE, AÑO DE CREACIÓN)
    """
    cabeceras = leer_todo(
        "purchase.requisition",
        [["company_id", "=", empresa_id], ["create_date", ">=", FECHA_DESDE]],
        ["name", "state", "periodo_id", "tipo_requerimiento", "importacion_check", "create_date", "fecha_aprobacion", "purchase_date_apro",
         "centro_costo", "requisition_origen_company_id", "state_calculatted_saved", "user_id"],
        [empresa_id])
    df_rq = pd.DataFrame([{
        "requisition_id": r["id"],
        "Cod_RQ": r["name"],
        "estado_rq": r["state"],
        "Estado_RQ": ESTADO_RQ.get(r["state"], r["state"]),
        "Tipo_RQ": r["tipo_requerimiento"],
        "Fecha_RQ": r["create_date"][:10],
        # "Fecha de Aprobación" del RQ: desde ahí es válido (un RQ creado antes puede reutilizarse y aprobarse después)
        "Fecha_Aprobacion": r["fecha_aprobacion"] or (r["purchase_date_apro"] or "")[:10] or None,
        "Anio": ANIO_FISCAL.get(m2o_id(r["periodo_id"])) or int(r["create_date"][:4]),
        "centro_rq": m2o_id(r["centro_costo"]),
        "rq_importacion": r["importacion_check"],
        "rq_hijo_de": m2o_nombre(r["requisition_origen_company_id"]),
        "Solicitante": m2o_nombre(r["user_id"]),
    } for r in cabeceras])

    """
        2. LÍNEAS DE RQ (CANTIDAD EN UdM DEL PRODUCTO)
    """
    lineas = leer_todo(
        "purchase.requisition.line",
        [["company_id", "=", empresa_id], ["requisition_id.create_date", ">=", FECHA_DESDE], ["product_id", "!=", False]],
        ["requisition_id", "product_id", "product_uom_id", "product_qty", "price_unit", "price_total", "account_analytic_id"],
        [empresa_id])
    filas = []
    for l in lineas:
        product_id = m2o_id(l["product_id"])
        cantidad, obs_udm = a_udm_producto(l["product_qty"] or 0.0, m2o_id(l["product_uom_id"]), product_id)
        filas.append({
            "rq_line_id": l["id"],
            "requisition_id": m2o_id(l["requisition_id"]),
            "product_id": product_id,
            "centro_linea_rq": m2o_id(l["account_analytic_id"]),
            "udm_rq": m2o_nombre(l["product_uom_id"]),
            "Cant_RQ": cantidad,
            "precio_rq_unit": l["price_unit"] or 0.0,
            "Valor_RQ_Soles": l["price_total"] or 0.0,   # "Precio en Soles" del RQ, con impuestos
            "obs_udm_rq": obs_udm,
        })
    df_lineas = pd.DataFrame(filas).merge(df_rq, on="requisition_id", how="left")
    return df_rq, df_lineas


requerimientos = {nombre: leer_requerimientos(empresa_id) for nombre, empresa_id in EMPRESAS.items()}
for nombre, (df_rq, df_rql) in requerimientos.items():
    print(f"{nombre}: {len(df_rq):,} RQ | {len(df_rql):,} líneas")

##### 3. Órdenes de compra, devoluciones y facturas de OC

In [ ]:
def leer_compras(empresa_id):
    """
        1. CABECERAS DE OC (SIN CANCELADAS)
    """
    ordenes = leer_todo(
        "purchase.order",
        [["company_id", "=", empresa_id], ["date_order", ">=", FECHA_DESDE], ["state", "!=", "cancel"]],
        ["name", "state", "state_new", "currency_id", "currency_rate", "date_approve", "date_order",
         "partner_id", "requisition_id", "origin", "importacion_check", "user_id"],
        [empresa_id])
    df_oc = pd.DataFrame([{
        "order_id": o["id"],
        "Cod_OC": o["name"],
        "estado_oc": o["state"],
        "estado_oc_new": o["state_new"],
        "Estado_OC": ESTADO_OC.get(o["state_new"], o["state_new"]),
        "Moneda": m2o_nombre(o["currency_id"]),
        "tasa": o["currency_rate"] if o["currency_rate"] else 1.0,
        "Fecha_OC": (o["date_approve"] or o["date_order"])[:10],
        "Proveedor": m2o_nombre(o["partner_id"]),
        "proveedor_id": m2o_id(o["partner_id"]),
        "oc_requisition_id": m2o_id(o["requisition_id"]),
        "oc_importacion": o["importacion_check"],
        "Comprador": m2o_nombre(o["user_id"]),           # "Representante de Compra" de la OC
    } for o in ordenes])

    """
        2. LÍNEAS DE OC
    """
    lineas = leer_todo(
        "purchase.order.line",
        [["company_id", "=", empresa_id], ["order_id.date_order", ">=", FECHA_DESDE], ["state", "!=", "cancel"], ["product_id", "!=", False]],
        ["order_id", "product_id", "product_uom", "product_qty", "qty_received", "qty_invoiced",
         "price_total", "account_analytic_id", "requisition_line_newid", "qty_received_method", "date_planned"],
        [empresa_id])
    df_ocl = pd.DataFrame([{
        "po_line_id": l["id"],
        "order_id": m2o_id(l["order_id"]),
        "product_id": m2o_id(l["product_id"]),
        "udm_oc_id": m2o_id(l["product_uom"]),
        "udm_oc": m2o_nombre(l["product_uom"]),
        "cant_oc_linea": l["product_qty"] or 0.0,
        "recibido_linea": l["qty_received"] or 0.0,
        "facturado_linea": l["qty_invoiced"] or 0.0,
        "total_linea": l["price_total"] or 0.0,          # en moneda de la OC, con impuestos
        "centro_oc": m2o_id(l["account_analytic_id"]),
        "rq_line_id": m2o_id(l["requisition_line_newid"]),
        "metodo_recepcion": l["qty_received_method"],
        "Fecha_Entrega": (l["date_planned"] or "")[:10] or None,
    } for l in lineas]).merge(df_oc, on="order_id", how="left")

    """
        3. DEVOLUCIONES A PROVEEDOR (MOVIMIENTOS HECHOS HACIA UBICACIÓN DE PROVEEDOR)
           to_refund = True -> Odoo ya las restó de lo recibido; False -> se restan aquí
    """
    devoluciones = leer_todo(
        "stock.move",
        [["company_id", "=", empresa_id], ["state", "=", "done"], ["purchase_line_id", "!=", False],
         ["location_dest_id.usage", "=", "supplier"], ["date", ">=", FECHA_DESDE]],
        ["purchase_line_id", "product_uom_qty", "product_uom", "to_refund", "picking_id"],
        [empresa_id])
    df_dev = pd.DataFrame([{
        "po_line_id": m2o_id(d["purchase_line_id"]),
        "cant_dev_linea": d["product_uom_qty"] or 0.0,
        "udm_dev": m2o_id(d["product_uom"]),
        "dev_restada_por_odoo": d["to_refund"],
        "albaran_dev": m2o_nombre(d["picking_id"]),
    } for d in devoluciones])

    """
        4. FACTURAS DE PROVEEDOR LIGADAS A OC (MONTO EN SOLES CON IMPUESTOS Y FECHA DE EMISIÓN)
    """
    apuntes = leer_todo(
        "account.move.line",
        [["company_id", "=", empresa_id], ["purchase_line_id", "!=", False], ["parent_state", "=", "posted"],
         ["exclude_from_invoice_tab", "=", False], ["date", ">=", FECHA_DESDE]],
        ["purchase_line_id", "move_id", "balance", "price_subtotal", "price_total", "date", "quantity", "product_uom_id", "product_id"],
        [empresa_id])
    asientos = {a["id"]: a for a in leer_ids("account.move", {m2o_id(a["move_id"]) for a in apuntes},
                                             ["name", "ref", "invoice_date", "move_type"], [empresa_id])}
    filas = []
    for a in apuntes:
        asiento = asientos.get(m2o_id(a["move_id"]), {})
        if asiento.get("move_type") not in ("in_invoice", "in_refund", "in_receipt"):
            continue
        ratio = (a["price_total"] / a["price_subtotal"]) if a["price_subtotal"] else 1.0
        filas.append({
            "po_line_id": m2o_id(a["purchase_line_id"]),
            "facturado_soles": (a["balance"] or 0.0) * ratio,
            "Factura": asiento.get("name"),
            "fecha_emision": asiento.get("invoice_date"),
            "anio_emision": int((asiento.get("invoice_date") or a["date"])[:4]),
            # Cantidad facturada publicada en UdM del producto (las NC restan). qty_invoiced de Odoo incluye borradores
            "cant_fact_prod": a_udm_producto(a["quantity"] or 0.0, m2o_id(a["product_uom_id"]), m2o_id(a["product_id"]))[0]
                              * (-1 if asiento.get("move_type") == "in_refund" else 1),
        })
    df_fact = pd.DataFrame(filas, columns=["po_line_id", "facturado_soles", "Factura", "fecha_emision", "anio_emision", "cant_fact_prod"])
    return df_ocl, df_dev, df_fact


compras = {nombre: leer_compras(empresa_id) for nombre, empresa_id in EMPRESAS.items()}
for nombre, (df_ocl, df_dev, df_fact) in compras.items():
    print(f"{nombre}: {df_ocl['order_id'].nunique():,} OC | {len(df_ocl):,} líneas | {len(df_dev):,} devoluciones | {len(df_fact):,} líneas de factura")

##### 4. Gasto sin OC (facturas directas, rendiciones y fondo fijo)

In [ ]:
def leer_gasto_sin_oc(empresa_id):
    """
        1. MOVIMIENTOS QUE ESTÁN DENTRO DE UNA RENDICIÓN
    """
    rendiciones = leer_todo("account.surrender.requests.it", [["company_id", "=", empresa_id]], ["name", "move_ids"], [empresa_id])
    asiento_rendicion = {m: r["name"] for r in rendiciones for m in r["move_ids"]}

    """
        1b. DEVOLUCIONES FÍSICAS A PROVEEDOR DE COMPRAS CON OC (PRODUCTO + PROVEEDOR -> OC)
            Si el proveedor emite la nota de crédito sin ligarla a la OC, esa NC no es un gasto negativo:
            la devolución ya se restó de lo recibido de la OC, en el año del RQ.
    """
    df_ocl, df_dev, _ = compras[nombre_empresa[empresa_id]]
    oc_de_devolucion = {}
    if not df_dev.empty:
        cruce = df_dev.merge(df_ocl[["po_line_id", "product_id", "proveedor_id", "Cod_OC"]], on="po_line_id", how="inner")
        for (p, prov), s in cruce.groupby(["product_id", "proveedor_id"])["Cod_OC"]:
            oc_de_devolucion[(p, prov)] = ", ".join(sorted(set(s)))
    ids_nc = set(odoo("account.move.line", "search", [[
        ["company_id", "=", empresa_id], ["parent_state", "=", "posted"], ["purchase_line_id", "=", False],
        ["product_id", "!=", False], ["move_id.move_type", "=", "in_refund"],
        ["date", ">=", f"{min(ANIOS)}-01-01"], ["date", "<=", f"{max(ANIOS)}-12-31"]]], {}, [empresa_id]))

    """
        2. LÍNEAS DE FACTURA DE PROVEEDOR SIN OC, CON PRODUCTO (POR FECHA CONTABLE)
    """
    apuntes = leer_todo(
        "account.move.line",
        [["company_id", "=", empresa_id], ["parent_state", "=", "posted"], ["purchase_line_id", "=", False],
         ["product_id", "!=", False], ["exclude_from_invoice_tab", "=", False],
         ["move_id.move_type", "in", ["in_invoice", "in_refund", "in_receipt"]],
         ["date", ">=", f"{min(ANIOS)}-01-01"], ["date", "<=", f"{max(ANIOS)}-12-31"]],
        ["product_id", "analytic_account_id", "balance", "price_subtotal", "price_total", "date", "move_id", "partner_id", "account_id",
         "quantity", "product_uom_id"],
        [empresa_id])
    movs_nc = {m2o_id(a["move_id"]) for a in apuntes if a["id"] in ids_nc}
    emision_nc = {m["id"]: m["invoice_date"] for m in leer_ids("account.move", movs_nc, ["invoice_date"], [empresa_id])}
    filas, filas_nc = [], []
    for a in apuntes:
        if PRODUCTO.get(m2o_id(a["product_id"]), {}).get("Cod_Producto") in PRODUCTOS_EXCLUIDOS:
            continue
        ratio = (a["price_total"] / a["price_subtotal"]) if a["price_subtotal"] else 1.0
        oc_dev = oc_de_devolucion.get((m2o_id(a["product_id"]), m2o_id(a["partner_id"])))
        if a["id"] in ids_nc and oc_dev:
            filas_nc.append({
                "Periodo": int((emision_nc.get(m2o_id(a["move_id"])) or a["date"])[:4]),   # año de emisión, como Odoo
                "product_id": m2o_id(a["product_id"]),
                "centro_id": m2o_id(a["analytic_account_id"]),
                "NC_Devolucion": (a["balance"] or 0.0) * ratio,
                "cuenta_id": m2o_id(a["account_id"]),
                "obs": f"NC {m2o_nombre(a['move_id'])} por devolución de {oc_dev}: no se resta aquí, ya se descontó en la compra (año del RQ)",
            })
            continue
        rendicion = asiento_rendicion.get(m2o_id(a["move_id"]))
        filas.append({
            "Periodo": int(a["date"][:4]),
            "Mes": int(a["date"][5:7]),
            "product_id": m2o_id(a["product_id"]),
            "centro_id": m2o_id(a["analytic_account_id"]),
            "Cuenta Analítica": m2o_nombre(a["analytic_account_id"]),
            "cuenta_id": m2o_id(a["account_id"]),
            "Gasto_Sin_OC": (a["balance"] or 0.0) * ratio,
            # Cantidad en UdM del producto, con el signo del importe (nota de crédito = negativa)
            "Cant_Sin_OC": a_udm_producto(abs(a["quantity"] or 0.0), m2o_id(a["product_uom_id"]), m2o_id(a["product_id"]))[0]
                           * (-1 if (a["balance"] or 0.0) < 0 else 1),
            "Gasto_Rendicion": (a["balance"] or 0.0) * ratio if rendicion else 0.0,
            "obs": "Gasto sin RQ/OC por rendición" if rendicion else "Gasto sin RQ/OC por factura directa",
        })
    nc_devolucion[nombre_empresa[empresa_id]] = pd.DataFrame(filas_nc, columns=["Periodo", "product_id", "centro_id", "NC_Devolucion", "cuenta_id", "obs"])
    return pd.DataFrame(filas, columns=["Periodo", "Mes", "product_id", "centro_id", "Cuenta Analítica", "cuenta_id",
                                        "Gasto_Sin_OC", "Cant_Sin_OC", "Gasto_Rendicion", "obs"])


nombre_empresa = {empresa_id: nombre for nombre, empresa_id in EMPRESAS.items()}
nc_devolucion = {}
gasto_sin_oc = {nombre: leer_gasto_sin_oc(empresa_id) for nombre, empresa_id in EMPRESAS.items()}
for nombre, df in gasto_sin_oc.items():
    print(nombre, df.groupby("Periodo")[["Gasto_Sin_OC", "Gasto_Rendicion"]].sum().round(0).to_dict("index"),
          f"| NC por devolución no restadas: {len(nc_devolucion[nombre])} líneas, S/ {nc_devolucion[nombre]['NC_Devolucion'].sum():,.0f}")

##### 4b. Facturado según Odoo (lo que usa la redistribución)

Odoo calcula el "Facturado Anual" de la redistribución con **todas** las facturas de proveedor del SKU + centro (con OC, sin OC y rendiciones), **con IGV** y por **año de emisión** de la factura. No resta RQ ni OC pendientes ni lo recibido sin facturar. Verificado contra redistribuciones reales (G1010460 2025 = 129 727,87; 2026 = 29 632,45).


In [ ]:
def leer_facturado_odoo(empresa_id):
    """
        FACTURAS DE PROVEEDOR POR AÑO DE EMISIÓN, CON IGV (MISMO CRITERIO QUE LA REDISTRIBUCIÓN DE ODOO)
    """
    filas = []
    for anio in ANIOS:
        apuntes = leer_todo(
            "account.move.line",
            [["company_id", "=", empresa_id], ["parent_state", "=", "posted"], ["product_id", "!=", False],
             ["exclude_from_invoice_tab", "=", False], ["move_id.move_type", "in", ["in_invoice", "in_refund", "in_receipt"]],
             ["move_id.invoice_date", ">=", f"{anio}-01-01"], ["move_id.invoice_date", "<=", f"{anio}-12-31"]],
            ["product_id", "analytic_account_id", "balance", "price_subtotal", "price_total", "account_id"],
            [empresa_id])
        for a in apuntes:
            if PRODUCTO.get(m2o_id(a["product_id"]), {}).get("Cod_Producto") in PRODUCTOS_EXCLUIDOS:
                continue
            ratio = (a["price_total"] / a["price_subtotal"]) if a["price_subtotal"] else 1.0
            filas.append({
                "Periodo": anio,
                "product_id": m2o_id(a["product_id"]),
                "centro_id": m2o_id(a["analytic_account_id"]),
                "cuenta_id": m2o_id(a["account_id"]),
                "Facturado_Odoo": (a["balance"] or 0.0) * ratio,
            })
    df = pd.DataFrame(filas, columns=["Periodo", "product_id", "centro_id", "cuenta_id", "Facturado_Odoo"])
    return df.groupby(["Periodo", "product_id", "centro_id"], dropna=False)["Facturado_Odoo"].sum().reset_index()


facturado_odoo = {nombre: leer_facturado_odoo(empresa_id) for nombre, empresa_id in EMPRESAS.items()}
for nombre, df in facturado_odoo.items():
    print(nombre, df.groupby("Periodo")["Facturado_Odoo"].sum().round(0).to_dict())


##### 5. Cálculo por línea de OC (ejecutado y en tránsito con OC)

- **Bien:** ejecutado = recibido neto de devoluciones. La parte facturada se valora con la factura (precio y TC reales); lo recibido sin factura, con el precio y la tasa de la OC. **Servicio** (producto tipo servicio, recepción manual, OC `SERV-` o RQ de tipo servicio; hay servicios mensuales mal configurados como almacenables): ejecutado = facturado, porque nunca se recepciona. La cantidad facturada se toma solo de facturas **publicadas** (Odoo cuenta también los borradores).
- Las facturas se separan por año de emisión para explicar la diferencia con lo que muestra Odoo.
- **Pendiente con OC:** pedido − ejecutado, solo si la OC sigue abierta (`Orden de Compra`, `Bloqueado`, `Cerrado Parcialmente`). Si está `Cerrado` o `Cerrada No Completada` el pendiente es 0 y se anota.

In [ ]:
def calcular_oc(nombre):
    df_rq, df_rql = requerimientos[nombre]
    df_ocl, df_dev, df_fact = compras[nombre]
    df = df_ocl.copy()

    """
        1. RQ DE CADA LÍNEA (POR LÍNEA DE RQ O, SI NO, POR EL RQ DE LA OC + PRODUCTO)
    """
    rq_por_linea = df_rql.set_index("rq_line_id")[["requisition_id", "centro_linea_rq", "udm_rq", "obs_udm_rq"]].to_dict("index")
    rq_por_cabecera = df_rq.set_index("requisition_id").to_dict("index")
    df["requisition_id"] = [
        rq_por_linea.get(r, {}).get("requisition_id") if pd.notna(r) else o
        for r, o in zip(df["rq_line_id"], df["oc_requisition_id"])
    ]
    df["requisition_id"] = df["requisition_id"].fillna(df["oc_requisition_id"])
    for campo in ["Cod_RQ", "Estado_RQ", "Tipo_RQ", "Anio", "centro_rq", "rq_hijo_de", "Fecha_RQ"]:
        df[campo] = df["requisition_id"].map(lambda r: rq_por_cabecera.get(r, {}).get(campo) if pd.notna(r) else None)
    df["udm_rq"] = df["rq_line_id"].map(lambda r: rq_por_linea.get(r, {}).get("udm_rq") if pd.notna(r) else None)

    """
        2. AÑO Y CENTRO DE COSTO
    """
    df["Periodo"] = df["Anio"].fillna(df["Fecha_OC"].str[:4].astype(int)).astype(int)
    centro_linea_rq = df["rq_line_id"].map(lambda r: rq_por_linea.get(r, {}).get("centro_linea_rq") if pd.notna(r) else None)
    df["centro_id"] = df["centro_oc"].fillna(centro_linea_rq).fillna(df["centro_rq"])

    """
        3. CANTIDADES EN UdM DEL PRODUCTO
    """
    conv = [a_udm_producto(1.0, u, p) for u, p in zip(df["udm_oc_id"], df["product_id"])]
    df["factor_udm"] = [c[0] for c in conv]
    df["obs_udm"] = [c[1] for c in conv]

    # Devoluciones que Odoo no restó (to_refund = False)
    if not df_dev.empty:
        df_dev = df_dev.copy()
        # Devolución en la UdM de la línea de OC (puede venir en otra UdM)
        udm_linea = df.set_index("po_line_id")["udm_oc_id"]
        df_dev["cant_dev_linea"] = [
            c / UDM[u]["factor"] * UDM[l]["factor"] if u in UDM and l in UDM and u != l and UDM[u]["categoria"] == UDM[l]["categoria"] else c
            for c, u, l in zip(df_dev["cant_dev_linea"], df_dev["udm_dev"], df_dev["po_line_id"].map(udm_linea))]
        no_restadas = df_dev[~df_dev["dev_restada_por_odoo"]].groupby("po_line_id")["cant_dev_linea"].sum()
        todas = df_dev.groupby("po_line_id").agg(cant_dev=("cant_dev_linea", "sum"),
                                                 albaranes_dev=("albaran_dev", lambda s: ", ".join(sorted(set(s.dropna())))))
        df["dev_no_restada"] = df["po_line_id"].map(no_restadas).fillna(0.0)
        df = df.merge(todas, left_on="po_line_id", right_index=True, how="left")
    else:
        df["dev_no_restada"], df["cant_dev"], df["albaranes_dev"] = 0.0, np.nan, None

    # Servicio: producto tipo servicio, recepción manual, OC "SERV-" o RQ de tipo servicio
    # (hay servicios mensuales configurados como almacenables, ej. G3180114 en SERV-00000535)
    df["es_servicio"] = ((df["product_id"].map(lambda p: PRODUCTO.get(p, {}).get("Tipo_Producto")) == "service")
                         | (df["metodo_recepcion"] == "manual")
                         | df["Cod_OC"].fillna("").str.startswith("SERV-")
                         | (df["Tipo_RQ"] == "servicio"))
    df["Cant_OC"] = df["cant_oc_linea"] * df["factor_udm"]
    df["Cant_Recibida"] = (df["recibido_linea"] - df["dev_no_restada"]) * df["factor_udm"]
    cant_facturada = df_fact.groupby("po_line_id")["cant_fact_prod"].sum() if not df_fact.empty else pd.Series(dtype="float64")
    df["Cant_Facturada"] = df["po_line_id"].map(cant_facturada).fillna(0.0)   # solo facturas publicadas
    df["Cant_Ejecutada"] = np.where(df["es_servicio"], df["Cant_Facturada"], df["Cant_Recibida"])

    """
        4. PRECIO UNITARIO EN SOLES CON IMPUESTOS (TOTAL DE LÍNEA / CANTIDAD / TASA DE LA OC)
    """
    df["precio_soles"] = np.where(df["Cant_OC"] > 0, df["total_linea"] / df["tasa"] / df["Cant_OC"].where(df["Cant_OC"] > 0, 1), 0.0)

    """
        5. FACTURAS: MONTO EN SOLES Y FECHA DE EMISIÓN
    """
    if not df_fact.empty:
        fact = df_fact.groupby("po_line_id").agg(
            facturado_soles=("facturado_soles", "sum"),
            Facturas=("Factura", lambda s: ", ".join(sorted(set(s.dropna())))),
            fechas_emision=("fecha_emision", lambda s: sorted(set(x for x in s if x))))
        df = df.merge(fact, left_on="po_line_id", right_index=True, how="left")
    else:
        df["facturado_soles"], df["Facturas"], df["fechas_emision"] = np.nan, None, None
    df["facturado_soles"] = df["facturado_soles"].fillna(0.0)

    # Facturas por año de emisión (Odoo las imputa a ese año; aquí van al año del RQ)
    if not df_fact.empty:
        por_anio = df_fact.groupby(["po_line_id", "anio_emision"])["facturado_soles"].sum().reset_index()
        periodo_linea = df.set_index("po_line_id")["Periodo"]
        por_anio["Periodo"] = por_anio["po_line_id"].map(periodo_linea)
        mismo = por_anio[por_anio["anio_emision"] == por_anio["Periodo"]].groupby("po_line_id")["facturado_soles"].sum()
        df["fact_mismo_anio"] = df["po_line_id"].map(mismo).fillna(0.0)
    else:
        por_anio = pd.DataFrame(columns=["po_line_id", "anio_emision", "facturado_soles", "Periodo"])
        df["fact_mismo_anio"] = 0.0
    df["fact_otro_anio"] = df["facturado_soles"] - df["fact_mismo_anio"]
    facturas_por_anio[nombre] = por_anio

    """
        6. EJECUTADO, PEDIDO Y PENDIENTE EN SOLES
    """
    confirmada = df["estado_oc"].isin(["purchase", "done"])
    abierta = confirmada & df["estado_oc_new"].isin(["purchase", "done", "parcialmente_cerrado"])
    df["Valor_OC"] = np.where(confirmada, df["Cant_OC"] * df["precio_soles"], 0.0)
    # Bien: lo facturado con la factura (en proporción a lo recibido) + lo recibido sin factura a precio de la OC
    cant_fact = df["Cant_Facturada"].clip(lower=0)
    cant_rec = df["Cant_Recibida"].clip(lower=0)
    proporcion = np.where(cant_fact > TOLERANCIA, np.minimum(cant_rec / cant_fact.where(cant_fact > TOLERANCIA, 1), 1.0), 0.0)
    valor_bien = df["facturado_soles"] * proporcion + (cant_rec - cant_fact).clip(lower=0) * df["precio_soles"]
    # Si el monto facturado no corresponde a la cantidad facturada (UdM o cantidades mal registradas en la factura),
    # no se puede repartir por cantidad: se valora lo recibido a precio de la OC (±15 % cubre diferencias de TC)
    valor_fact_a_precio_oc = cant_fact * df["precio_soles"]
    relacion = df["facturado_soles"] / valor_fact_a_precio_oc.where(valor_fact_a_precio_oc.abs() > TOLERANCIA, np.nan)
    df["factura_coherente"] = (cant_fact <= TOLERANCIA) | relacion.between(0.85, 1.15)
    valor_bien = np.where(df["factura_coherente"], valor_bien, cant_rec * df["precio_soles"])
    valor_servicio = np.where(df["facturado_soles"] != 0, df["facturado_soles"], df["Cant_Ejecutada"] * df["precio_soles"])
    df["Valor_Ejecutado"] = np.where(confirmada, np.where(df["es_servicio"], valor_servicio, valor_bien), 0.0)
    df["Cant_Pendiente"] = np.where(abierta, (df["Cant_OC"] - df["Cant_Ejecutada"]).clip(lower=0), 0.0)
    df["Cant_Pendiente"] = np.where(df["Cant_Pendiente"] < TOLERANCIA, 0.0, df["Cant_Pendiente"])
    df["Valor_Pendiente"] = df["Cant_Pendiente"] * df["precio_soles"]
    df["confirmada"] = confirmada

    """
        7. OBSERVACIONES
    """
    def observar(r):
        obs = []
        if r["Moneda"] and r["Moneda"] != "PEN":
            tc = 1 / r["tasa"] if r["tasa"] else 0
            obs.append(f"OC en {r['Moneda']} (TC {tc:.3f})")
        if r["oc_importacion"]:
            obs.append("Importación")
        if r["obs_udm"]:
            obs.append(f"UdM OC: {r['obs_udm']}")
        if isinstance(r["udm_rq"], str) and isinstance(r["udm_oc"], str) and r["udm_rq"] != r["udm_oc"]:
            obs.append(f"Pedido en RQ con {r['udm_rq']} y en OC con {r['udm_oc']}")
        if pd.notna(r["cant_dev"]) and r["cant_dev"]:
            obs.append(f"Devolución a proveedor {r['cant_dev'] * r['factor_udm']:g} ({r['albaranes_dev']})")
        if r["es_servicio"]:
            obs.append("Servicio: avance por facturación")
        elif r["confirmada"] and not r["factura_coherente"]:
            obs.append("Monto facturado no corresponde a la cantidad facturada: valorado a precio de OC")
        elif r["confirmada"] and r["Cant_Recibida"] - r["Cant_Facturada"] > TOLERANCIA:
            obs.append(f"Recibido sin facturar {r['Cant_Recibida'] - r['Cant_Facturada']:g}: valorado a precio de OC")
        elif r["confirmada"] and r["Cant_Facturada"] - r["Cant_Recibida"] > TOLERANCIA:
            dif = r["Cant_Facturada"] - r["Cant_Recibida"]
            if pd.notna(r["cant_dev"]) and r["cant_dev"]:
                obs.append(f"Facturado y luego devuelto al proveedor ({dif:g}): la compra se cuenta neta de la devolución en el año del RQ")
            elif r["Cant_Pendiente"] > 0:
                obs.append(f"Facturado sin recibir {dif:g}: queda como pendiente con OC")
            else:
                obs.append(f"Facturado sin recibir {dif:g} y OC cerrada: no se cuenta")
        if r["Cant_Ejecutada"] > r["Cant_OC"] + TOLERANCIA:
            obs.append("Ejecutado mayor a lo pedido")
        if r["confirmada"] and r["estado_oc_new"] == "cerrado_no" and r["Cant_OC"] - r["Cant_Ejecutada"] > TOLERANCIA:
            obs.append(f"OC cerrada no completada: {r['Cant_OC'] - r['Cant_Ejecutada']:g} sin ejecutar que no se cuentan")
        if r["Cant_Pendiente"] > 0 and (pd.Timestamp.today() - pd.Timestamp(r["Fecha_OC"])).days > 365:
            obs.append("OC abierta con más de 12 meses: revisar si se debe cerrar")
        if r["confirmada"] and r["estado_oc_new"] == "cerrado" and r["Cant_OC"] - r["Cant_Ejecutada"] > TOLERANCIA:
            obs.append(f"OC cerrada con {r['Cant_OC'] - r['Cant_Ejecutada']:g} sin ejecutar")
        if isinstance(r["fechas_emision"], list):
            otros = [f for f in r["fechas_emision"] if int(f[:4]) != r["Periodo"]]
            if otros:
                obs.append(f"Factura emitida el {', '.join(otros)}: Odoo la imputa a {', '.join(sorted({f[:4] for f in otros}))}, aquí va a {r['Periodo']}")
        if isinstance(r["rq_hijo_de"], str) and r["rq_hijo_de"]:
            obs.append(f"RQ relacionado de {r['rq_hijo_de']}")
        if r["Proveedor"] and r["Proveedor"].startswith(("TATSA PERU", "TRANSVIAL LIMA")):
            obs.append("OC a empresa relacionada")
        if pd.isna(r["requisition_id"]):
            obs.append("OC sin RQ")
        return " | ".join(obs)

    df["Observaciones"] = df.apply(observar, axis=1)
    return df


facturas_por_anio = {}
calculo_oc = {nombre: calcular_oc(nombre) for nombre in EMPRESAS}
for nombre, df in calculo_oc.items():
    c = df[df["confirmada"]]
    print(f"{nombre}: OC confirmadas {c['order_id'].nunique():,} | pendientes con OC {(c['Cant_Pendiente'] > 0).sum():,} líneas")

##### 6. RQ sin OC (en tránsito sin OC)

Líneas de RQ aprobadas (`Confirmado`, `En proceso`, `Selección del licitador`) menos lo ya pedido en OC confirmadas.
- **Servicios:** la cantidad del RQ no se puede comparar con la de la OC (el RQ repite el servicio varias veces y la OC lo agrupa en 1 por el mismo monto). Si el RQ de servicio ya tiene OC confirmada, **no queda pendiente**; solo cuentan los RQ de servicio sin ninguna OC.
- **Precio referencial:** solo si el RQ trae precio **exactamente 1** (típico de importaciones) o total 0. Ahí se usa el último precio de compra en soles. Un precio real bajo (ej. S/ 0,80 por kg) se respeta.


In [ ]:
def calcular_rq_sin_oc(nombre):
    df_rq, df_rql = requerimientos[nombre]
    df_oc = calculo_oc[nombre]

    """
        1. LO YA PEDIDO EN OC CONFIRMADAS POR LÍNEA DE RQ (O POR RQ + PRODUCTO SI LA OC NO GUARDÓ LA LÍNEA)
    """
    confirmadas = df_oc[df_oc["confirmada"]]
    pedido_linea = confirmadas.dropna(subset=["rq_line_id"]).groupby("rq_line_id")["Cant_OC"].sum()
    pedido_rq_prod = confirmadas[confirmadas["rq_line_id"].isna()].groupby(["requisition_id", "product_id"])["Cant_OC"].sum()
    cotizaciones = df_oc[~df_oc["confirmada"]].groupby("requisition_id")["Cod_OC"].agg(lambda s: ", ".join(sorted(set(s))))

    """
        2. ÚLTIMO PRECIO DE COMPRA EN SOLES POR PRODUCTO (PARA RQ CON PRECIO REFERENCIAL)
    """
    ultimo_precio = (confirmadas[confirmadas["precio_soles"] > 0].sort_values("Fecha_OC")
                     .groupby("product_id")["precio_soles"].last())

    df = df_rql[df_rql["estado_rq"].isin(["in_progress", "ongoing", "open"]) & df_rql["Anio"].isin(ANIOS)].copy()
    df["Cant_Pedida_OC"] = df["rq_line_id"].map(pedido_linea).fillna(0.0)
    # OC sin línea de RQ: lo pedido se reparte entre las líneas del RQ con ese producto, en orden y sin pasar su cantidad
    df = df.sort_values("rq_line_id").copy()
    restante = defaultdict(float, pedido_rq_prod.to_dict())
    repartido = []
    for r, p, cant, ya in zip(df["requisition_id"], df["product_id"], df["Cant_RQ"], df["Cant_Pedida_OC"]):
        tomar = min(restante[(r, p)], max(cant - ya, 0.0))
        restante[(r, p)] -= tomar
        repartido.append(tomar)
    df["Cant_Pedida_OC"] += repartido
    df["Cant_Pendiente"] = (df["Cant_RQ"] - df["Cant_Pedida_OC"]).clip(lower=0)
    df["es_servicio"] = (df["Tipo_RQ"] == "servicio") | (df["product_id"].map(lambda p: PRODUCTO.get(p, {}).get("Tipo_Producto")) == "service")
    # Servicio con OC confirmada para ese RQ + producto (aunque el RQ repita el servicio en varias líneas):
    # la diferencia de cantidad entre RQ y OC no es un pendiente real
    rq_prod_con_oc = set(zip(confirmadas["requisition_id"], confirmadas["product_id"]))
    con_oc = [(r, p) in rq_prod_con_oc for r, p in zip(df["requisition_id"], df["product_id"])]
    df.loc[df["es_servicio"] & ((df["Cant_Pedida_OC"] > 0) | pd.Series(con_oc, index=df.index)), "Cant_Pendiente"] = 0.0
    df = df[df["Cant_Pendiente"] > TOLERANCIA].copy()

    """
        3. VALOR EN SOLES CON IMPUESTOS
    """
    precio_rq = np.where(df["Cant_RQ"] > 0, df["Valor_RQ_Soles"] / df["Cant_RQ"].where(df["Cant_RQ"] > 0, 1), 0.0)
    referencial = np.isclose(df["precio_rq_unit"], 1.0) | (precio_rq <= 0)
    df["precio_soles"] = np.where(referencial, df["product_id"].map(ultimo_precio).fillna(0.0), precio_rq)
    df["Valor_Pendiente"] = df["Cant_Pendiente"] * df["precio_soles"]

    def observar(r, ref):
        obs = []
        if r["obs_udm_rq"]:
            obs.append(f"UdM RQ: {r['obs_udm_rq']}")
        if ref:
            obs.append("Precio del RQ referencial: se usó el último precio de compra" if r["precio_soles"] > 0 else "Precio del RQ referencial y sin compras previas: valor 0")
        if r["Cant_Pedida_OC"] > 0:
            obs.append(f"Ya pedido en OC {r['Cant_Pedida_OC']:g} de {r['Cant_RQ']:g}")
        cot = cotizaciones.get(r["requisition_id"])
        if cot:
            obs.append(f"Cotización en curso: {cot}")
        if r["rq_importacion"]:
            obs.append("Importación")
        if r["Tipo_RQ"] == "servicio":
            obs.append("Servicio: avance por facturación")
        if isinstance(r["rq_hijo_de"], str) and r["rq_hijo_de"]:
            obs.append(f"RQ relacionado de {r['rq_hijo_de']}")
        return " | ".join(obs)

    df["Observaciones"] = [observar(r, ref) for (_, r), ref in zip(df.iterrows(), referencial)]
    df["centro_id"] = df["centro_linea_rq"].fillna(df["centro_rq"])
    df["Periodo"] = df["Anio"].astype(int)
    return df


rq_sin_oc = {nombre: calcular_rq_sin_oc(nombre) for nombre in EMPRESAS}
for nombre, df in rq_sin_oc.items():
    print(f"{nombre}: {len(df):,} líneas de RQ sin OC | S/ {df['Valor_Pendiente'].sum():,.0f}")

##### 7. Saldo disponible por SKU, centro de costo y año

In [ ]:
COLUMNAS_SALDO = [
    "Visible_Odoo", "Cod_Grupo_Sku", "Cod_Producto", "Producto", "Nro_Parte", "Marca", "Familia", "Cuenta Analítica",
    "Partida", "Situación presupuestaria", "Varias_Situaciones", "Ppto_por_Situacion", "Periodo", "Presupuesto", "Compras", "Compras_und", "RQ", "RQ_und",
    "RQ_COC", "RQ_COC_und", "RQ_SOC", "RQ_SOC_und", "Gasto_Sin_OC", "Gasto_Rendicion", "Saldo",
    "Facturado_Odoo", "Disponible_Odoo", "Disponible_Redistribuir", "Diferencia_vs_Odoo",
    "Dif_Factura_Otro_Anio", "Dif_Recibido_vs_Facturado", "Dif_NC_Devolucion", "Dif_Otros", "Alerta_Redistribucion",
    "Por_Facturacion", "Observaciones",
]

COMPONENTES_DIF = {
    "RQ_SOC": "RQ sin OC (comprometido que Odoo no ve)",
    "RQ_COC": "OC pendiente de entrega/facturación (comprometido que Odoo no ve)",
    "Dif_Factura_Otro_Anio": "Factura emitida en otro año que el del RQ",
    "Dif_Recibido_vs_Facturado": "Recibido sin facturar / facturado antes de recibir / devuelto al proveedor",
    "Dif_NC_Devolucion": "Nota de crédito por devolución de una compra de otro año",
    "Dif_Otros": "Otros (gasto sin OC por fecha, centro distinto, TC)",
}


def armar_saldo(nombre):
    claves = CLAVES
    df_oc = calculo_oc[nombre]
    df_oc = df_oc[df_oc["confirmada"] & df_oc["Periodo"].isin(ANIOS)]

    """
        1. CADA FUENTE AGRUPADA A SKU + CENTRO + AÑO
    """
    oc = df_oc.groupby(claves, dropna=False).agg(
        Compras=("Valor_Ejecutado", "sum"), Compras_und=("Cant_Ejecutada", "sum"),
        fact_lineas=("facturado_soles", "sum"), fact_a_otro_anio=("fact_otro_anio", "sum"),
        RQ=("Valor_OC", "sum"), RQ_und=("Cant_OC", "sum"),
        RQ_COC=("Valor_Pendiente", "sum"), RQ_COC_und=("Cant_Pendiente", "sum"),
        serv_oc=("es_servicio", "any"), obs_oc=("Observaciones", unir_observaciones)).reset_index()
    soc = rq_sin_oc[nombre].groupby(claves, dropna=False).agg(
        RQ_SOC=("Valor_Pendiente", "sum"), RQ_SOC_und=("Cant_Pendiente", "sum"),
        serv_soc=("es_servicio", "any"), obs_soc=("Observaciones", unir_observaciones)).reset_index()
    gasto = gasto_sin_oc[nombre]
    gasto = gasto[gasto["Periodo"].isin(ANIOS)].groupby(claves, dropna=False).agg(
        Gasto_Sin_OC=("Gasto_Sin_OC", "sum"), Gasto_Rendicion=("Gasto_Rendicion", "sum"),
        obs_gasto=("obs", unir_observaciones)).reset_index()
    pres = INFO_SITUACION[nombre]

    # Facturas emitidas en el año que pertenecen a RQ/OC de otro año (Odoo las resta aquí, el saldo real no)
    lineas = calculo_oc[nombre]
    lineas = lineas[lineas["confirmada"]].set_index("po_line_id")[["product_id", "centro_id"]]
    ajenas = facturas_por_anio[nombre]
    ajenas = ajenas[(ajenas["anio_emision"] != ajenas["Periodo"]) & ajenas["anio_emision"].isin(ANIOS)]
    ajenas = ajenas.join(lineas, on="po_line_id", how="inner")
    ajenas = (ajenas.assign(Periodo=ajenas["anio_emision"]).groupby(claves, dropna=False)["facturado_soles"].sum()
              .rename("fact_de_otro_anio").reset_index())
    fact_odoo = facturado_odoo[nombre]
    nc = nc_devolucion[nombre]
    nc = nc[nc["Periodo"].isin(ANIOS)].groupby(claves, dropna=False).agg(
        NC_Devolucion=("NC_Devolucion", "sum"), obs_nc=("obs", unir_observaciones)).reset_index()

    """
        2. UNIÓN DE TODAS LAS FUENTES (CLAVES NUMÉRICAS PARA QUE EL CRUCE NO FALLE CON VACÍOS)
    """
    def normalizar(parte):
        parte = parte.copy()
        for clave in claves:
            parte[clave] = pd.to_numeric(parte[clave], errors="coerce").astype("float64")
        return parte

    df = normalizar(pres)
    for parte in [oc, soc, gasto, ajenas, fact_odoo, nc]:
        df = df.merge(normalizar(parte), on=claves, how="outer")
    df["Periodo"] = df["Periodo"].astype(int)

    medidas = ["Presupuesto", "Compras", "Compras_und", "RQ", "RQ_und", "RQ_COC", "RQ_COC_und", "RQ_SOC", "RQ_SOC_und",
               "Gasto_Sin_OC", "Gasto_Rendicion", "fact_lineas", "fact_a_otro_anio", "fact_de_otro_anio", "Facturado_Odoo", "NC_Devolucion"]
    df[medidas] = df[medidas].fillna(0.0)
    df["Saldo"] = df["Presupuesto"] - (df["Compras"] + df["RQ_COC"] + df["RQ_SOC"] + df["Gasto_Sin_OC"])

    """
        2b. COMPARACIÓN CON LO QUE ODOO DEJA REDISTRIBUIR
    """
    df["Disponible_Odoo"] = df["Presupuesto"] - df["Facturado_Odoo"]
    df["Disponible_Redistribuir"] = df["Saldo"].clip(lower=0)
    df["Diferencia_vs_Odoo"] = df["Disponible_Odoo"] - df["Saldo"]
    df["Dif_Factura_Otro_Anio"] = df["fact_a_otro_anio"] - df["fact_de_otro_anio"]
    df["Dif_Recibido_vs_Facturado"] = df["Compras"] - df["fact_lineas"]
    df["Dif_NC_Devolucion"] = -df["NC_Devolucion"]
    df["Dif_Otros"] = df["Diferencia_vs_Odoo"] - (df["RQ_SOC"] + df["RQ_COC"] + df["Dif_Factura_Otro_Anio"]
                                                  + df["Dif_Recibido_vs_Facturado"] + df["Dif_NC_Devolucion"])

    def alerta(r):
        if abs(r["Diferencia_vs_Odoo"]) < UMBRAL_ALERTA:
            return ""
        motivo = max(COMPONENTES_DIF, key=lambda c: abs(r[c]))
        if r["Saldo"] < 0 and r["Disponible_Odoo"] > 0:
            texto = f"Odoo deja redistribuir {r['Disponible_Odoo']:,.2f} pero está sobregirado {r['Saldo']:,.2f}"
        elif r["Diferencia_vs_Odoo"] > 0:
            texto = f"Odoo muestra {r['Diferencia_vs_Odoo']:,.2f} más de lo real"
        else:
            texto = f"Odoo muestra {-r['Diferencia_vs_Odoo']:,.2f} menos de lo real"
        return f"{texto}. Motivo principal: {COMPONENTES_DIF[motivo]}"

    df["Alerta_Redistribucion"] = df.apply(alerta, axis=1)
    df["Por_Facturacion"] = np.where(df[["serv_oc", "serv_soc"]].fillna(False).any(axis=1), "Sí", "No")

    def observaciones(r):
        partes = [r.get("obs_oc"), r.get("obs_soc"), r.get("obs_gasto"), r.get("obs_nc")]
        if r["Presupuesto"] == 0 and (r["Compras"] + r["RQ_COC"] + r["RQ_SOC"] + r["Gasto_Sin_OC"]) != 0 and abs(r["Saldo"]) > TOLERANCIA:
            partes.append("Sin presupuesto para este SKU y centro")
        if pd.isna(r["centro_id"]):
            partes.append("Sin centro de costo")
        return unir_observaciones(pd.Series(partes))

    df["Observaciones"] = df.apply(observaciones, axis=1)

    """
        3. DATOS DEL PRODUCTO Y DEL CENTRO
    """
    centros = {}
    for fuente in [presupuesto[nombre], gasto_sin_oc[nombre]]:
        if fuente.empty:
            continue
        centros.update(fuente.dropna(subset=["centro_id"]).drop_duplicates("centro_id").set_index("centro_id")["Cuenta Analítica"].to_dict())
    faltan = [int(c) for c in df["centro_id"].dropna().unique() if c not in centros]
    if faltan:
        centros.update({c["id"]: c["display_name"] for c in leer_ids("account.analytic.account", faltan, ["display_name"], [1, 2])})
    df["Cuenta Analítica"] = df["centro_id"].map(centros)
    for columna, vacio in [("Partida", "Sin presupuesto"), ("Situación presupuestaria", "Sin presupuesto"),
                           ("Varias_Situaciones", "No"), ("Ppto_por_Situacion", "")]:
        df[columna] = df[columna].fillna(vacio)
    visible = df["Visible_Odoo"].astype("object").where(df["Visible_Odoo"].notna(), df["centro_id"].map(lambda c: clave_id(c) in CENTROS_VISIBLES[nombre]))
    df["Visible_Odoo"] = np.where(visible.astype(bool), "Sí", "No")
    df = df.merge(df_productos.drop(columns=["Tipo_Producto", "udm_producto"]), on="product_id", how="left")

    df = df[COLUMNAS_SALDO].sort_values(["Periodo", "Cod_Producto", "Cuenta Analítica"]).reset_index(drop=True)
    return df


saldo = {nombre: armar_saldo(nombre) for nombre in EMPRESAS}
df_saldo_tatsa, df_saldo_tvl = saldo["TATSA"], saldo["TVL"]
df_saldo_tatsa.head()

##### 8. Resumen por centro de costo y situación presupuestaria

In [ ]:
def resumen(df):
    return (df.groupby(["Periodo", "Cuenta Analítica"], dropna=False)
              [["Presupuesto", "Compras", "RQ_COC", "RQ_SOC", "Gasto_Sin_OC", "Saldo", "Facturado_Odoo", "Disponible_Odoo", "Diferencia_vs_Odoo"]].sum()
              .round(2).reset_index())


df_resumen = pd.concat([resumen(df).assign(Empresa=nombre) for nombre, df in saldo.items()], ignore_index=True)
df_resumen = df_resumen[["Empresa"] + [c for c in df_resumen.columns if c != "Empresa"]]
pd.set_option("display.float_format", "{:,.2f}".format)
df_resumen.groupby(["Empresa", "Periodo"])[["Presupuesto", "Compras", "RQ_COC", "RQ_SOC", "Gasto_Sin_OC", "Saldo", "Disponible_Odoo", "Diferencia_vs_Odoo"]].sum()

##### 8b. Alertas para redistribución

SKU + centro donde lo que Odoo deja redistribuir (`Disponible_Odoo`) difiere del saldo real en más de `UMBRAL_ALERTA`. **Antes de ceder presupuesto, usar `Disponible_Redistribuir`, no lo que muestra Odoo.** Ordenado por el exceso que Odoo dejaría mover.


In [ ]:
COLUMNAS_ALERTAS = [
    "Empresa", "Periodo", "Visible_Odoo", "Cod_Producto", "Producto", "Cuenta Analítica", "Partida", "Situación presupuestaria",
    "Presupuesto", "Facturado_Odoo", "Disponible_Odoo",
    "Saldo", "Disponible_Redistribuir", "Diferencia_vs_Odoo", "RQ_SOC", "RQ_COC", "Dif_Factura_Otro_Anio",
    "Dif_Recibido_vs_Facturado", "Dif_NC_Devolucion", "Dif_Otros", "Alerta_Redistribucion", "Observaciones",
]

df_alertas = pd.concat([df.assign(Empresa=nombre) for nombre, df in saldo.items()], ignore_index=True)
df_alertas = df_alertas[df_alertas["Diferencia_vs_Odoo"].abs() >= UMBRAL_ALERTA][COLUMNAS_ALERTAS]
df_alertas = df_alertas.sort_values(["Periodo", "Diferencia_vs_Odoo"], ascending=[False, False]).reset_index(drop=True)

# Cuánto deja Odoo redistribuir de más (solo donde el saldo real es menor), por empresa y año
exceso = df_alertas[df_alertas["Diferencia_vs_Odoo"] > 0]
exceso.groupby(["Empresa", "Periodo"]).agg(
    SKU_centro=("Cod_Producto", "size"),
    Odoo_deja_de_mas=("Diferencia_vs_Odoo", "sum"),
    por_RQ_SOC=("RQ_SOC", "sum"), por_RQ_COC=("RQ_COC", "sum"),
    por_factura_otro_anio=("Dif_Factura_Otro_Anio", "sum"),
    por_recibido_sin_factura=("Dif_Recibido_vs_Facturado", "sum"),
    sobregirados_que_odoo_ve_con_saldo=("Alerta_Redistribucion", lambda s: int(s.str.contains("sobregirado").sum())))


##### 9. Pendientes: OC por entregar y RQ sin OC

- **OC pendiente de entrega:** bienes en OC abiertas con entrega parcial o sin entrega. Muestra RQ, OC, repuesto, comprador (*Representante de Compra* de la OC), días desde la OC y días de atraso frente a la fecha de entrega.
- **RQ sin OC:** RQ aprobados que aún no tienen OC confirmada. `Dias_RQ` se cuenta desde la **Fecha de Aprobación** del RQ (`fecha_aprobacion`), que es cuando el RQ es válido: un RQ puede crearse un mes y aprobarse (reutilizarse) después. Si no tiene aprobación, se usa la fecha de creación (`Fecha_RQ`). En el Excel se dejan solo los centros visibles en Odoo (`SOLO_VISIBLES_RQ_SIN_OC`).

Los servicios pendientes de facturar no van aquí (no se entregan); siguen sumando en `RQ_COC` del saldo.


In [ ]:
HOY = pd.Timestamp.today().normalize()

COLUMNAS_OC_PENDIENTE = [
    "Empresa", "Periodo", "Visible_Odoo", "Cod_RQ", "Fecha_RQ", "Cod_OC", "Fecha_OC", "Dias_OC", "Comprador", "Proveedor",
    "Cod_Producto", "Producto", "Cuenta Analítica", "Partida", "Situación presupuestaria", "Cant_OC", "Cant_Recibida", "Cant_Pendiente",
    "Estado_Entrega", "Fecha_Entrega_Prevista", "Dias_Atraso", "Valor_Pendiente", "Estado_OC", "Observaciones",
]
COLUMNAS_RQ_SIN_OC = [
    "Empresa", "Periodo", "Visible_Odoo", "Cod_RQ", "Estado_RQ", "Tipo_RQ", "Fecha_RQ", "Fecha_Aprobacion", "Dias_RQ", "Solicitante",
    "Cod_Producto", "Producto", "Cuenta Analítica", "Partida", "Situación presupuestaria", "Cant_RQ", "Cant_Pedida_OC", "Cant_Pendiente",
    "Valor_Pendiente", "Observaciones",
]


def nombres_centro(nombre, centros):
    # Nombre del centro de costo para una serie de ids
    mapa = {}
    for fuente in [presupuesto[nombre], gasto_sin_oc[nombre]]:
        if not fuente.empty:
            mapa.update(fuente.dropna(subset=["centro_id"]).drop_duplicates("centro_id").set_index("centro_id")["Cuenta Analítica"].to_dict())
    faltan = [int(c) for c in centros.dropna().unique() if c not in mapa]
    if faltan:
        mapa.update({c["id"]: c["display_name"] for c in leer_ids("account.analytic.account", faltan, ["display_name"], [1, 2])})
    return centros.map(mapa)


def completar(nombre, df):
    # Columnas comunes: empresa, visible, situación, producto y centro
    df["Empresa"] = nombre
    centro_visible = df["centro_id"].map(lambda c: clave_id(c) in CENTROS_VISIBLES[nombre])
    df["Cuenta Analítica"] = nombres_centro(nombre, df["centro_id"])
    # Visible_Odoo igual que en SALDO: la línea de presupuesto del SKU + centro + año; si no tiene presupuesto, el centro
    info = INFO_SITUACION[nombre][CLAVES + ["Partida", "Situación presupuestaria", "Visible_Odoo"]].rename(columns={"Visible_Odoo": "vis_linea"})
    df = claves_float(df.assign(centro_visible=centro_visible.values)).merge(info, on=CLAVES, how="left")
    df["Visible_Odoo"] = np.where(df["vis_linea"].astype("object").where(df["vis_linea"].notna(), df["centro_visible"]).astype(bool), "Sí", "No")
    df[["Partida", "Situación presupuestaria"]] = df[["Partida", "Situación presupuestaria"]].fillna("Sin presupuesto")
    df["Periodo"] = df["Periodo"].astype(int)
    return df.merge(df_productos[["product_id", "Cod_Producto", "Producto"]], on="product_id", how="left")


def armar_oc_pendiente(nombre):
    """
        BIENES EN OC ABIERTAS SIN ENTREGA O CON ENTREGA PARCIAL
    """
    df = calculo_oc[nombre]
    df = df[df["confirmada"] & ~df["es_servicio"] & (df["Cant_Pendiente"] > 0) & df["Periodo"].isin(ANIOS)].copy()
    df["Estado_Entrega"] = np.where(df["Cant_Recibida"] > TOLERANCIA, "Entrega parcial", "Sin entrega")
    df["Dias_OC"] = (HOY - pd.to_datetime(df["Fecha_OC"])).dt.days
    prevista = pd.to_datetime(df["Fecha_Entrega"], errors="coerce")
    df["Fecha_Entrega_Prevista"] = prevista.dt.strftime("%Y-%m-%d")
    df["Dias_Atraso"] = (HOY - prevista).dt.days.clip(lower=0)
    df = completar(nombre, df)
    return df[COLUMNAS_OC_PENDIENTE].sort_values(["Dias_OC", "Cod_OC"], ascending=[False, True]).reset_index(drop=True)


def armar_rq_sin_oc(nombre):
    """
        RQ APROBADOS SIN OC CONFIRMADA (POR LA CANTIDAD PENDIENTE)
    """
    df = rq_sin_oc[nombre].copy()
    # Días desde la Fecha de Aprobación (si el RQ no la tiene, desde su creación)
    df["Fecha_Aprobacion"] = df["Fecha_Aprobacion"].fillna(df["Fecha_RQ"])
    df["Dias_RQ"] = (HOY - pd.to_datetime(df["Fecha_Aprobacion"])).dt.days
    df = completar(nombre, df)
    return df[COLUMNAS_RQ_SIN_OC].sort_values(["Dias_RQ", "Cod_RQ"], ascending=[False, True]).reset_index(drop=True)


df_oc_pendiente = pd.concat([armar_oc_pendiente(nombre) for nombre in EMPRESAS], ignore_index=True)
df_rq_sin_oc = pd.concat([armar_rq_sin_oc(nombre) for nombre in EMPRESAS], ignore_index=True)

print("OC pendientes de entrega:")
print(df_oc_pendiente.groupby(["Empresa", "Estado_Entrega"]).agg(Lineas=("Cod_OC", "size"), OC=("Cod_OC", "nunique"),
      Valor=("Valor_Pendiente", "sum"), Dias_promedio=("Dias_OC", "mean")).round(0).to_string())
print("\nRQ sin OC (todos / visibles):")
df_rq_sin_oc.groupby(["Empresa", "Visible_Odoo"]).agg(Lineas=("Cod_RQ", "size"), RQ=("Cod_RQ", "nunique"), Valor=("Valor_Pendiente", "sum")).round(0)


##### 9b. Gasto por mes (control mensual y proyección)

El mismo gasto del saldo (**Compras + RQ_COC + RQ_SOC + Gasto_Sin_OC**) repartido por el **mes en que se consumió el presupuesto**:
- RQ con OC y RQ sin OC: mes de creación del RQ (si la OC no tiene RQ, mes de la OC). Si el RQ se creó en otro año que su periodo de presupuesto, va a enero (creado antes) o diciembre (creado después).
- Gasto sin OC: mes de la fecha contable.

`Fuente` usa los nombres de las columnas del saldo: **Compras** (lo ejecutado), **RQ_COC** (pendiente con OC), **RQ_SOC** (RQ sin OC) y **Gasto_Sin_OC**. Filtrando `Fuente = Compras` se obtienen las compras de la hoja SALDO / RESUMEN. `Cantidad` está en la UdM del producto y cuadra con `Compras_und`, `RQ_COC_und` y `RQ_SOC_und` del saldo (en servicios es la cantidad facturada o pendiente de facturar, referencial; en gasto sin OC, la cantidad de la factura, negativa si es nota de crédito).

Por SKU + centro + año, la suma de los meses es igual al gasto del saldo (control abajo). Lo usa `Proyeccion_Presupuesto.ipynb` para el flujo mensual del saldo.

In [ ]:
def mes_del_gasto(fecha, periodo):
    # Mes en que el RQ consumió presupuesto; RQ creado en otro año que su periodo: enero (antes) o diciembre (después)
    if not isinstance(fecha, str) or not fecha:
        return 12
    anio, mes = int(fecha[:4]), int(fecha[5:7])
    return mes if anio == periodo else 1 if anio < periodo else 12


def armar_gasto_mes(nombre):
    oc = calculo_oc[nombre]
    oc = oc[oc["confirmada"] & oc["Periodo"].isin(ANIOS)]
    soc = rq_sin_oc[nombre]
    sin_oc = gasto_sin_oc[nombre]
    sin_oc = sin_oc[sin_oc["Periodo"].isin(ANIOS)]
    meses_oc = [mes_del_gasto(f, p) for f, p in zip(oc["Fecha_RQ"].fillna(oc["Fecha_OC"]), oc["Periodo"])]
    partes = [
        pd.DataFrame({"Periodo": oc["Periodo"], "product_id": oc["product_id"], "centro_id": oc["centro_id"],
                      "Mes": meses_oc, "Fuente": "Compras", "Cantidad": oc["Cant_Ejecutada"], "Gasto": oc["Valor_Ejecutado"]}),
        pd.DataFrame({"Periodo": oc["Periodo"], "product_id": oc["product_id"], "centro_id": oc["centro_id"],
                      "Mes": meses_oc, "Fuente": "RQ_COC", "Cantidad": oc["Cant_Pendiente"], "Gasto": oc["Valor_Pendiente"]}),
        pd.DataFrame({"Periodo": soc["Periodo"], "product_id": soc["product_id"], "centro_id": soc["centro_id"],
                      "Mes": [mes_del_gasto(f, p) for f, p in zip(soc["Fecha_RQ"], soc["Periodo"])],
                      "Fuente": "RQ_SOC", "Cantidad": soc["Cant_Pendiente"], "Gasto": soc["Valor_Pendiente"]}),
        pd.DataFrame({"Periodo": sin_oc["Periodo"], "product_id": sin_oc["product_id"], "centro_id": sin_oc["centro_id"],
                      "Mes": sin_oc["Mes"], "Fuente": "Gasto_Sin_OC", "Cantidad": sin_oc["Cant_Sin_OC"], "Gasto": sin_oc["Gasto_Sin_OC"]}),
    ]
    df = pd.concat([claves_float(p) for p in partes], ignore_index=True)
    df = df.groupby(CLAVES + ["Mes", "Fuente"], dropna=False)[["Cantidad", "Gasto"]].sum().reset_index()
    df = df[(df["Gasto"].abs() > TOLERANCIA) | (df["Cantidad"].abs() > TOLERANCIA)]
    df = completar(nombre, df)
    return df[["Empresa", "Periodo", "Mes", "Visible_Odoo", "Cod_Producto", "Producto", "Cuenta Analítica", "Partida",
               "Situación presupuestaria", "Fuente", "Cantidad", "Gasto"]]


df_gasto_mes = pd.concat([armar_gasto_mes(nombre) for nombre in EMPRESAS], ignore_index=True)

# Control: por SKU + centro + año, la suma de los meses = gasto del saldo, y cada fuente = su columna del saldo
llave = ["Empresa", "Periodo", "Cod_Producto", "Cuenta Analítica"]
del_saldo = pd.concat([d.assign(Empresa=n) for n, d in saldo.items()], ignore_index=True)
del_saldo = del_saldo.assign(Gasto=del_saldo[["Compras", "RQ_COC", "RQ_SOC", "Gasto_Sin_OC"]].sum(axis=1)).groupby(llave, dropna=False)["Gasto"].sum()
por_mes = df_gasto_mes.groupby(llave, dropna=False)["Gasto"].sum()
dif = por_mes.sub(del_saldo, fill_value=0.0).abs()
print(f"Gasto por mes: {len(df_gasto_mes):,} filas | cuadra con el saldo: {bool((dif < 0.01).all())} (máx. diferencia S/ {dif.max():,.2f})")
for fuente in ["Compras", "RQ_COC", "RQ_SOC", "Gasto_Sin_OC"]:
    a = df_gasto_mes[df_gasto_mes["Fuente"] == fuente].groupby(llave, dropna=False)["Gasto"].sum()
    b = pd.concat([d.assign(Empresa=n) for n, d in saldo.items()], ignore_index=True).groupby(llave, dropna=False)[fuente].sum()
    print(f"  {fuente}: cuadra con la columna del saldo: {bool((a.sub(b, fill_value=0.0).abs() < 0.01).all())}")
for fuente, columna in [("Compras", "Compras_und"), ("RQ_COC", "RQ_COC_und"), ("RQ_SOC", "RQ_SOC_und")]:
    a = df_gasto_mes[df_gasto_mes["Fuente"] == fuente].groupby(llave, dropna=False)["Cantidad"].sum()
    b = pd.concat([d.assign(Empresa=n) for n, d in saldo.items()], ignore_index=True).groupby(llave, dropna=False)[columna].sum()
    print(f"  {fuente}: cantidad cuadra con {columna}: {bool((a.sub(b, fill_value=0.0).abs() < 0.001).all())}")
df_gasto_mes[df_gasto_mes["Periodo"] == max(ANIOS)].pivot_table(index="Empresa", columns="Mes", values="Gasto", aggfunc="sum").round(0)

##### 10. Controles

In [ ]:
"""
    1. EL PRESUPUESTO DEL SALDO CUADRA CON LO LEÍDO DE ODOO
"""
for nombre in EMPRESAS:
    leido = presupuesto[nombre].groupby("Periodo")["Presupuesto"].sum()
    en_saldo = saldo[nombre].groupby("Periodo")["Presupuesto"].sum()
    print(nombre, "presupuesto cuadra:", bool(np.allclose(leido.reindex(ANIOS, fill_value=0), en_saldo.reindex(ANIOS, fill_value=0))))


"""
    2. CASOS DE REFERENCIA: G1010460 (IMPORTACIÓN CON FACTURA EMITIDA EN EL AÑO ANTERIOR)
                            G1280028 (COMPRA 2025 DEVUELTA EN 2026 CON NC SIN OC)
"""
df_saldo_tatsa[df_saldo_tatsa["Cod_Producto"].isin(["G1010460", "G1280028"]) & (df_saldo_tatsa["Cuenta Analítica"].str.contains("TA01432", na=False))][
    ["Cod_Producto"] +
    ["Cuenta Analítica", "Periodo", "Presupuesto", "Compras", "Compras_und", "RQ_COC", "RQ_SOC", "Gasto_Sin_OC", "Saldo",
     "Facturado_Odoo", "Disponible_Odoo", "Diferencia_vs_Odoo", "Dif_Factura_Otro_Anio", "Dif_Recibido_vs_Facturado", "Dif_NC_Devolucion", "Alerta_Redistribucion"]]

In [ ]:
"""
    3. FILAS CON OBSERVACIONES POR TIPO
"""
tipos = {
    "Factura emitida en otro año": "Factura emitida",
    "UdM distinta / convertida": "UdM",
    "Devolución a proveedor": "Devolución",
    "Servicio (por facturación)": "Servicio",
    "Precio RQ referencial": "referencial",
    "Gasto por rendición": "rendición",
    "Sin presupuesto": "Sin presupuesto",
    "OC cerrada no completada": "no completada",
    "OC abierta con más de 12 meses": "más de 12 meses",
}
pd.DataFrame({nombre: {t: int(df["Observaciones"].str.contains(p, regex=False).sum()) for t, p in tipos.items()}
              for nombre, df in saldo.items()})

In [ ]:
"""
    4. RECEPCIONES EN USD A REGULARIZAR EN EL KARDEX
       TC = 1: el kardex muestra dólares como soles. Sin factura: el kardex usa el TC del día de recepción hasta que se vincule.
       (El saldo no depende del kardex; esto es para pedir la regularización antes del cierre.)
"""
filas = []
for nombre, empresa_id in EMPRESAS.items():
    movimientos = leer_todo(
        "stock.move",
        [["company_id", "=", empresa_id], ["state", "=", "done"], ["location_id.usage", "=", "supplier"],
         ["purchase_line_id.order_id.currency_id.name", "=", "USD"], ["date", ">=", FECHA_DESDE]],
        ["picking_id", "purchase_line_id"], [empresa_id])
    albaranes = {m2o_id(m["picking_id"]) for m in movimientos if m["picking_id"]}
    for p in leer_ids("stock.picking", albaranes, ["name", "origin", "tc", "invoice_id", "landed_cost_id", "date_done"], [empresa_id]):
        if p["tc"] and p["tc"] >= 2 and p["invoice_id"]:
            continue
        filas.append({
            "Empresa": nombre, "Albaran": p["name"], "OC": p["origin"], "Fecha": (p["date_done"] or "")[:10],
            "TC": p["tc"], "Factura": m2o_nombre(p["invoice_id"]), "GV": m2o_nombre(p["landed_cost_id"]),
            "Problema": "TC = 1: kardex en dólares" if not p["tc"] or p["tc"] < 2 else "Sin factura vinculada",
        })
df_recepciones_revisar = pd.DataFrame(filas, columns=["Empresa", "Albaran", "OC", "Fecha", "TC", "Factura", "GV", "Problema"])
df_recepciones_revisar = df_recepciones_revisar.sort_values(["Problema", "Fecha"], ascending=[False, True]).reset_index(drop=True)
print(df_recepciones_revisar.groupby(["Empresa", "Problema"]).size())
df_recepciones_revisar[df_recepciones_revisar["Problema"].str.startswith("TC")]


##### 11. Exportar a Excel (opcional)

Solo se ejecuta con `EXPORTAR_EXCEL = True` en Parámetros. El Excel es una versión **simplificada** (el detalle completo queda en los DataFrames del notebook):

| Hoja | Contenido |
|---|---|
| `RESUMEN` | Totales por empresa, año, centro de costo, partida y situación presupuestaria |
| `SALDO` | Una fila por empresa + año + SKU + centro (las dos empresas juntas); situaciones sumadas como en Odoo |
| `ALERTAS` | Solo donde Odoo muestra un disponible distinto al real, con el desglose de la diferencia |
| `OC_PENDIENTE_ENTREGA` | Bienes en OC abiertas sin entrega o con entrega parcial: RQ, OC, repuesto, comprador, días |
| `RQ_SIN_OC` | RQ aprobados sin OC (solo centros visibles si `SOLO_VISIBLES_RQ_SIN_OC = True`) |
| `RECEPCIONES_REVISAR` | Albaranes en USD con TC = 1 o sin factura vinculada (para regularizar el kardex) |
| `GASTO_MES` | Gasto del saldo por mes del RQ (o fecha contable si no hay OC): base del control mensual y de `Proyeccion_Presupuesto.ipynb` |

Todas las hojas tienen `Visible_Odoo` (lo que se veía inicialmente en Odoo) y `Partida` / `Situación presupuestaria` para segmentar. Diseño: sin cuadrícula, letra Consolas 8, importes en soles (negativos en rojo) y encabezados por color: azul = identificación, verde = presupuesto, ámbar = consumo/comprometido, rojo = saldo, morado = comparación con Odoo, azul acero = seguimiento (días/fechas), gris = notas. Alineación: centrado en columnas cortas (códigos, fechas, estados, días), derecha en importes y cantidades, y textos largos a la izquierda con ancho fijo (lo que no entra queda tapado por la columna siguiente). Montos, cantidades y columnas cortas con ancho automático. En `Observaciones` solo quedan las notas que requieren atención.


In [ ]:
from openpyxl.styles import Border

FUENTE = Font(name="Consolas", size=8)
FUENTE_TITULO = Font(name="Consolas", size=8, bold=True, color="FFFFFF")
# Textos largos: ancho fijo moderado; lo que no entra queda tapado por la columna siguiente
ANCHOS = {"Observaciones": 50, "Alerta": 50, "Ppto_por_Situacion": 40, "Producto": 34, "Situación presupuestaria": 30,
          "Cuenta Analítica": 30, "Proveedor": 24, "Comprador": 18, "Solicitante": 18, "Partida": 18, "Familia": 18}

# Formatos de número
FORMATO_SOLES = '"S/" #,##0.00;[Red]-"S/" #,##0.00'
FORMATO_CANTIDAD = "#,##0.00"
FORMATO_DIAS = "#,##0"
COLUMNAS_SOLES = {"Presupuesto", "Compras", "Gasto_Sin_OC", "RQ_COC", "RQ_SOC", "Saldo", "Facturado_Odoo", "Disponible_Odoo",
                  "Diferencia_vs_Odoo", "Dif_Factura_Otro_Anio", "Dif_Recibido_vs_Facturado", "Dif_NC_Devolucion", "Dif_Otros",
                  "Valor_Pendiente", "Gasto"}
COLUMNAS_DIAS = {"Dias_OC", "Dias_RQ", "Dias_Atraso"}

# Color del encabezado por grupo de columnas
COLOR_GRUPO = {
    "presupuesto": "375623",   # verde
    "consumo": "BF8F00",       # ámbar
    "saldo": "C00000",         # rojo oscuro
    "odoo": "7030A0",          # morado
    "seguimiento": "2E75B6",   # azul acero
    "notas": "595959",         # gris
    "identificacion": "1F4E78" # azul oscuro (por defecto)
}
GRUPO_COLUMNA = {
    "Presupuesto": "presupuesto",
    **{c: "consumo" for c in ["Compras", "Compras_und", "Gasto_Sin_OC", "RQ_COC", "RQ_COC_und", "RQ_SOC", "RQ_SOC_und", "Valor_Pendiente",
                              "Cant_OC", "Cant_Recibida", "Cant_Pendiente", "Cant_RQ", "Cant_Pedida_OC", "Gasto", "Cantidad"]},
    "Saldo": "saldo",
    **{c: "odoo" for c in ["Facturado_Odoo", "Disponible_Odoo", "Diferencia_vs_Odoo", "Dif_Factura_Otro_Anio",
                           "Dif_Recibido_vs_Facturado", "Dif_NC_Devolucion", "Dif_Otros"]},
    **{c: "seguimiento" for c in ["Dias_OC", "Dias_RQ", "Dias_Atraso", "Estado_Entrega", "Fecha_Entrega_Prevista",
                                  "Fecha_OC", "Fecha_RQ", "Fecha_Aprobacion", "Estado_OC", "Estado_RQ"]},
    **{c: "notas" for c in ["Alerta", "Observaciones", "Ppto_por_Situacion", "Problema"]},
}

# Alineación: centrado en columnas cortas, derecha en números, izquierda en textos largos (sin ajuste: se tapan con la siguiente)
COLUMNAS_CENTRADAS = {"Empresa", "Periodo", "Visible_Odoo", "Cod_Producto", "Varias_Situaciones", "Cod_RQ", "Cod_OC",
                      "Estado_RQ", "Tipo_RQ", "Fecha_RQ", "Fecha_Aprobacion", "Dias_RQ", "Fecha_OC", "Dias_OC", "Estado_Entrega", "Fecha_Entrega_Prevista",
                      "Dias_Atraso", "Estado_OC", "Albaran", "OC", "Fecha", "TC", "Factura", "GV", "Mes", "Fuente"}
ALINEA_CENTRO = Alignment(horizontal="center", vertical="center")
ALINEA_DERECHA = Alignment(horizontal="right", vertical="center")
ALINEA_TEXTO = Alignment(horizontal="left", vertical="center")


def ancho_automatico(df, columna, formato):
    # Ancho según el valor más largo ya formateado (montos, cantidades, días y columnas cortas)
    valores = df[columna].dropna()
    if formato == FORMATO_SOLES:
        largos = [len(f"S/ {v:,.2f}") + (1 if v < 0 else 0) for v in valores]
    elif formato == FORMATO_CANTIDAD:
        largos = [len(f"{v:,.2f}") for v in valores]
    elif formato == FORMATO_DIAS:
        largos = [len(f"{v:,.0f}") for v in valores]
    else:
        largos = [len(str(v)) for v in valores]
    dato = max(largos, default=0)
    return min(max(dato, len(str(columna)) * 0.85, 6) + 2, 30)


def dar_formato(hoja, df):
    # Sin cuadrícula, Consolas 8, encabezado con color por grupo, soles en importes, filtro y primera fila fija
    hoja.sheet_view.showGridLines = False
    for i, columna in enumerate(df.columns, start=1):
        celda = hoja.cell(row=1, column=i)
        celda.font = FUENTE_TITULO
        celda.fill = PatternFill("solid", fgColor=COLOR_GRUPO[GRUPO_COLUMNA.get(columna, "identificacion")])
        celda.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        celda.border = Border()
    formatos = {}
    for i, columna in enumerate(df.columns, start=1):
        if columna in COLUMNAS_SOLES:
            formatos[i] = FORMATO_SOLES
        elif columna in COLUMNAS_DIAS:
            formatos[i] = FORMATO_DIAS
        elif pd.api.types.is_float_dtype(df[columna]):
            formatos[i] = FORMATO_CANTIDAD
    for i, columna in enumerate(df.columns, start=1):
        hoja.column_dimensions[get_column_letter(i)].width = ANCHOS.get(columna) or ancho_automatico(df, columna, formatos.get(i))
    alineacion = {}
    for i, columna in enumerate(df.columns, start=1):
        if columna in COLUMNAS_CENTRADAS:
            alineacion[i] = ALINEA_CENTRO
        elif i in formatos or pd.api.types.is_numeric_dtype(df[columna]):
            alineacion[i] = ALINEA_DERECHA
        else:
            alineacion[i] = ALINEA_TEXTO
    for fila in hoja.iter_rows(min_row=2):
        for celda in fila:
            celda.font = FUENTE
            celda.alignment = alineacion[celda.column]
            if celda.column in formatos:
                celda.number_format = formatos[celda.column]
    hoja.row_dimensions[1].height = 24
    hoja.freeze_panes = "A2"
    hoja.auto_filter.ref = hoja.dimensions


NOTAS_INFORMATIVAS = ("OC en ", "Importación", "RQ relacionado de", "Servicio: avance por facturación")


def limpiar_observaciones(texto):
    # Deja solo las notas que piden revisar algo
    if not isinstance(texto, str):
        return ""
    return " | ".join(p for p in texto.split(" | ") if p and p != "nan" and not p.startswith(NOTAS_INFORMATIVAS))


def hoja_saldo():
    df = pd.concat([d.assign(Empresa=n) for n, d in saldo.items()], ignore_index=True)
    df = df.rename(columns={"Alerta_Redistribucion": "Alerta"})
    df["Observaciones"] = df["Observaciones"].map(limpiar_observaciones)
    columnas = ["Empresa", "Periodo", "Visible_Odoo", "Cod_Producto", "Producto", "Familia", "Cuenta Analítica",
                "Partida", "Situación presupuestaria", "Varias_Situaciones", "Presupuesto", "Compras", "Compras_und",
                "Gasto_Sin_OC", "RQ_COC", "RQ_COC_und", "RQ_SOC", "RQ_SOC_und", "Saldo", "Disponible_Odoo", "Diferencia_vs_Odoo", "Alerta",
                "Ppto_por_Situacion", "Observaciones"]
    # Se quitan filas sin ningún movimiento (presupuesto, consumo y unidades en tránsito en cero)
    medidas = ["Presupuesto", "Compras", "Gasto_Sin_OC", "RQ_COC", "RQ_COC_und", "RQ_SOC", "RQ_SOC_und", "Disponible_Odoo"]
    df = df[df[medidas].abs().sum(axis=1) > TOLERANCIA]
    return df[columnas].sort_values(["Empresa", "Periodo", "Cod_Producto", "Cuenta Analítica"]).reset_index(drop=True)


def hoja_resumen():
    df = pd.concat([d.assign(Empresa=n) for n, d in saldo.items()], ignore_index=True)
    return (df.groupby(["Empresa", "Periodo", "Visible_Odoo", "Cuenta Analítica", "Partida", "Situación presupuestaria"], dropna=False)
              [["Presupuesto", "Compras", "Gasto_Sin_OC", "RQ_COC", "RQ_SOC", "Saldo", "Disponible_Odoo", "Diferencia_vs_Odoo"]]
              .sum().round(2).reset_index())


def hoja_alertas():
    df = df_alertas.rename(columns={"Alerta_Redistribucion": "Alerta"})
    return df[["Empresa", "Periodo", "Visible_Odoo", "Cod_Producto", "Producto", "Cuenta Analítica", "Partida",
               "Situación presupuestaria", "Saldo", "Disponible_Odoo", "Diferencia_vs_Odoo", "RQ_SOC", "RQ_COC", "Dif_Factura_Otro_Anio",
               "Dif_Recibido_vs_Facturado", "Dif_NC_Devolucion", "Dif_Otros", "Alerta"]]


def hoja_oc_pendiente():
    df = df_oc_pendiente.copy()
    df["Observaciones"] = df["Observaciones"].map(limpiar_observaciones)
    return df


def hoja_rq_sin_oc():
    df = df_rq_sin_oc.copy()
    if SOLO_VISIBLES_RQ_SIN_OC:
        df = df[df["Visible_Odoo"] == "Sí"]
    df["Observaciones"] = df["Observaciones"].map(limpiar_observaciones)
    return df.reset_index(drop=True)


if EXPORTAR_EXCEL:
    hojas = {"RESUMEN": hoja_resumen(), "SALDO": hoja_saldo(), "ALERTAS": hoja_alertas(),
             "OC_PENDIENTE_ENTREGA": hoja_oc_pendiente(), "RQ_SIN_OC": hoja_rq_sin_oc(),
             "RECEPCIONES_REVISAR": df_recepciones_revisar, "GASTO_MES": df_gasto_mes}
    with pd.ExcelWriter(ARCHIVO_EXCEL, engine="openpyxl") as escritor:
        for nombre_hoja, df in hojas.items():
            df.to_excel(escritor, sheet_name=nombre_hoja, index=False)
            dar_formato(escritor.sheets[nombre_hoja], df)
    print(f"Excel generado: {ARCHIVO_EXCEL}", {h: len(d) for h, d in hojas.items()})
else:
    print("Exportación desactivada (EXPORTAR_EXCEL = False)")
